# Phase 3 — The full Transformer

Stack the Phase-2 components into the whole model and *check it makes sense before
training a single step*:

- `Encoder` / `Decoder` (N layers + final LayerNorm) and `Transformer` (embeddings +
  √d_model scaling + PE + weight tying + output projection) — all in
  `transformer/transformer.py`.
- **Param count** + where the parameters actually live.
- A **full forward pass on a real batch**, printing every intermediate shape.
- The key sanity check: **untrained loss ≈ log(vocab_size)**. If it isn't, the
  embedding init or the label-smoothing math is wrong — fix it before Phase 4.

In [ ]:
# Bootstrap: put the repo root on sys.path so `import transformer` works from notebooks/
import sys, pathlib
ROOT = pathlib.Path.cwd()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo root:", ROOT)

In [ ]:
import math
import torch
import torch.nn.functional as F

from transformer.config import ModelConfig
from transformer.tokenizer import train_joint_bpe, PAD_ID
from transformer.data import load_multi30k, make_dataloader
from transformer.transformer import Transformer

def get_device():
    if torch.cuda.is_available(): return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")
device = get_device()
torch.manual_seed(0)
print("device:", device)

## Build a real vocab + a real batch

We need the actual vocab size to size the model and to compute the log(V) reference.
Train the joint BPE on Multi30k (offline fallback if needed), then pull one real batch.

In [ ]:
data, online = load_multi30k()
print("online:", online, "| train pairs:", len(data["train"]))
tokenizer = train_joint_bpe(data["train"], vocab_size=10000)
VOCAB = tokenizer.get_vocab_size()
print("vocab size:", VOCAB)

loader = make_dataloader(data["train"], tokenizer, batch_size=16, shuffle=True, max_len=40)
batch = {k: v.to(device) for k, v in next(iter(loader)).items()}
print({k: tuple(v.shape) for k, v in batch.items()})

## Instantiate the model (smoke scale) and count parameters

We use the small `smoke()` config so a forward pass is instant on MPS/CPU. On the A100
you'd swap in the first-run config (`ModelConfig(...)` defaults) or `ModelConfig.base()`.

In [ ]:
cfg = ModelConfig.smoke(src_vocab_size=VOCAB, tgt_vocab_size=VOCAB)
model = Transformer(cfg).to(device)
print("config:", cfg)
print(f"\ntotal parameters: {model.num_parameters():,}")

## Where do the parameters live?

Break the count down by top-level block. Note **weight tying**: the source embedding,
target embedding, and output projection share ONE matrix, so the projection adds
*zero* new parameters. Inside each layer the FFN is the biggest block — about **2×**
the attention parameters (attention is 4 × d_model²; the FFN is 2 × d_model × d_ff with
d_ff = 4·d_model). The "4×" you hear about is the `d_ff/d_model` ratio, not the param ratio.

In [ ]:
def count(module):
    return sum(p.numel() for p in module.parameters())

emb = cfg.tgt_vocab_size * cfg.d_model
print(f"shared embedding matrix : {emb:,}  (src=tgt=output_proj, tied)")
print(f"encoder (all layers+LN) : {count(model.encoder):,}")
print(f"decoder (all layers+LN) : {count(model.decoder):,}")
print(f"output_proj (tied->free): {count(model.output_proj):,}  (shares the embedding)")
print(f"{'-'*42}")
print(f"unique total            : {model.num_parameters():,}")

# Per-encoder-layer: attention vs FFN.
L0 = model.encoder.layers[0]
print(f"\nper encoder layer  attn={count(L0.self_attn):,}  ffn={count(L0.ffn):,}"
      f"  (ffn/attn ratio ~ {count(L0.ffn)/count(L0.self_attn):.1f}x)")

## Full forward pass — print every intermediate shape

Trace the data through the network. The encoder produces `memory` (B, S, d_model); the
decoder consumes `memory` + the shifted target to produce (B, T, d_model); the output
projection maps that to logits over the vocab (B, T, V).

In [ ]:
model.eval()
with torch.no_grad():
    emb_src = model._embed(batch["src"], model.src_embed)
    print("src embedded         :", tuple(emb_src.shape))
    memory = model.encode(batch["src"], batch["src_pad"])
    print("encoder memory       :", tuple(memory.shape))
    decoded = model.decode(batch["tgt_in"], memory, batch["tgt_pad"], batch["src_pad"])
    print("decoder output       :", tuple(decoded.shape))
    logits = model.output_proj(decoded)
    print("logits (B, T, vocab) :", tuple(logits.shape))
    # full forward in one call should match:
    logits2 = model(batch["src"], batch["tgt_in"], batch["src_pad"], batch["tgt_pad"])
    assert torch.allclose(logits, logits2)
print("\nforward() matches the manual encode/decode/project path ✓")

## The sanity check: untrained loss ≈ log(vocab_size)

An untrained model outputs near-uniform logits, so softmax ≈ 1/V on every token. The
cross-entropy of a uniform prediction is exactly `-log(1/V) = log(V)` — and with label
smoothing it stays at `log(V)` (the smoothed target weights still sum to 1). So the
**very first loss should land near `log(V)`**. A wildly different value means the init,
the √d_model scaling, or the loss masking is off.

In [ ]:
with torch.no_grad():
    logits = model(batch["src"], batch["tgt_in"], batch["src_pad"], batch["tgt_pad"])
    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB),
        batch["tgt_out"].reshape(-1),
        ignore_index=PAD_ID,
        label_smoothing=0.1,
    )

ref = math.log(VOCAB)
print(f"untrained loss : {loss.item():.3f}")
print(f"log(vocab)     : {ref:.3f}")
print(f"ratio          : {loss.item()/ref:.3f}  (want ~1.0)")
assert 0.8 * ref < loss.item() < 1.2 * ref, "untrained loss far from log(V) — investigate!"
print("\nuntrained loss is at the log(V) reference ✓  — the model is initialized sanely.")

## Trace one token (do this on paper)

Pick token `tgt_in[0, 3]`. It is embedded (× √d_model), gets PE added, then in each
decoder layer it: (1) attends to `tgt_in[0, 0:4]` (causal — never 4+), (2) cross-attends
to the *entire* encoder memory of the source sentence, (3) passes through the FFN. After
the stack + final LN, `output_proj` turns its 128-dim vector into a distribution over
the vocab — the model's guess for token *4*. That's the whole network in one sentence.

## Takeaways

- The model is just shape-preserving blocks + an embedding in and a projection out.
- Weight tying makes the output projection free and couples embedding learning to the loss.
- Untrained loss ≈ log(V) is the cheapest, highest-value correctness check there is.

**Next:** `04_training.ipynb` — the Noam LR schedule, then the make-or-break
**overfit-one-batch** test (loss must approach ~0), then a real (small) training run
with live loss/LR/grad-norm curves.